## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual [here](https://quantra.quantinsti.com/quantra-notebook).
2. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
3. Before running this notebook on your local PC:
   - You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on **"Run Codes Locally on Your Machine"** in the course.
   - You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.


# Streaming Live Market Data with Zerodha Kite WebSocket

Welcome! This notebook is your hands-on guide to using the Zerodha Kite Connect WebSocket API. In the previous video, we discussed the theory of WebSockets. Now, we'll put that theory into practice.

Our goal is to connect to the WebSocket and get a live stream of price updates for a specific stock. We'll see every single price change (or 'tick') as it happens on the exchange, printed right here in our notebook.

## Outline

1.  **Setup & Initialization**: Importing libraries and setting up our WebSocket connector.
2.  **Understanding Callbacks**: Learning how to handle events like new data and connection changes.
3.  **Defining the Callback Functions**: Writing the Python functions to manage the live data feed.
4.  **Assigning Callbacks**: Telling our connector which function to run for each event.
5.  **Connecting and Streaming**: Starting the connection to receive live data.
6.  **Conclusion & Next Steps**: Recapping what we've learned and exploring what's next.

### 1. Setup & Initialization

Now, let's start writing our script. We'll import the `KiteTicker` class, which is specifically designed for WebSocket streaming. We'll also import the `logging` module to see status messages, which helps in understanding what's happening behind the scenes.

In [1]:
# Import the necessary libraries
from kiteconnect import KiteTicker
import logging

# --- Setup ---
# This basic configuration for logging will help us see the events in the console.
logging.basicConfig(level=logging.DEBUG)

# --- Your Credentials ---
# IMPORTANT: Replace these with your actual api_key and access_token.
# You get the access_token after a successful login via the Kite Connect API.
api_key = "YOUR_API_KEY"
access_token = "YOUR_ACCESS_TOKEN"

# --- Initialise the Ticker ---
# This creates our WebSocket 'ticker' object.
# Think of it as setting up the phone line before making a call.
kws = KiteTicker(api_key, access_token)

### 2. Understanding Callbacks

The most important concept for WebSockets is the **callback**. A callback is a function that we define and then assign to a specific event. The WebSocket client will automatically run our function whenever that event occurs. It's an event-driven way of handling a continuous stream of information.

We need to create functions for three key events:

-   `on_ticks`: This is the main one. It runs **every time we receive a price update** (a 'tick').
-   `on_connect`: This runs **once the connection is successfully established**.
-   `on_close`: This runs **if the connection is closed** for any reason.

### 3. Defining the Callback Functions

Let's write the three functions. Pay close attention to the comments, as they explain what each part does.

In [3]:
# --- Callback Functions ---

def on_ticks(ws, ticks):
    """
    This function is called every time a new tick is received.
    'ticks' is a list of tick data dictionaries.
    """
    # In a real trading algorithm, this is where your strategy logic would go.
    logging.info(f"Ticks: {ticks}")

def on_connect(ws, response):
    """
    This function runs once the connection is established.
    We use this to subscribe to the instruments we want to track.
    """
    logging.info("Successfully connected! WebSocket connection is now open.")
    
    # --- Subscribe to Instruments ---
    # We need to tell the server which instruments we are interested in.
    # For this example, we'll subscribe to Reliance (NSE instrument token: 738561).
    # You can subscribe to multiple instruments by adding more tokens to the list.
    instrument_tokens = [738561]
    ws.subscribe(instrument_tokens)
    
    # --- Set the Subscription Mode ---
    # We can also set the 'mode' for the ticks we want to receive.
    # 'MODE_FULL': Gives everything: open, high, low, close, volume, OI, depth, etc.
    # 'MODE_QUOTE': Gives open, high, low, close, last price, and volume.
    # 'MODE_LTP': Gives us just the Last Traded Price. This is the most lightweight.
    ws.set_mode(ws.MODE_FULL, instrument_tokens)
    logging.info(f"Subscribed to tokens {instrument_tokens} in {ws.MODE_FULL} mode.")

def on_close(ws, code, reason):
    """
    This function runs when the connection is closed.
    """
    logging.info(f"Connection closed: {code} - {reason}")
    # You might want to add logic here to try and reconnect.
    # For example, create a loop or a function to automatically attempt reconnection.
    ws.stop()

### 4. Assigning Callbacks

Now that we have our functions, we need to tell our `kws` object about them. We assign our functions to the corresponding `on_` properties of the `kws` object.

In [4]:
# --- Assigning Callbacks ---
# Now we tell our 'kws' object which function to call for which event.
kws.on_ticks = on_ticks
kws.on_connect = on_connect
kws.on_close = on_close

### 5. Connecting and Streaming

Everything is set up! The final step is to start the connection.

The `kws.connect()` method starts the WebSocket client. It's a **blocking call**, which means it will run forever, continuously listening for messages from the server until the connection is closed or the script is stopped. When you run the cell below, you will see the log messages appear as events happen.

In [5]:
# --- Start the Connection ---
# This is a blocking call, which means it will run forever until it's stopped.
# To stop it, you will need to interrupt the kernel in your Jupyter Notebook.
print("Connecting to WebSocket... (This will run indefinitely. Interrupt the kernel to stop.)")
kws.connect()

INFO:root:Successfully connected! WebSocket connection is now open.
INFO:root:Subscribed to tokens [738561] in full mode.


Connecting to WebSocket... (This will run indefinitely. Interrupt the kernel to stop.)


INFO:root:Ticks: [{'tradable': True, 'mode': 'full', 'instrument_token': 738561, 'last_price': 1480.8, 'last_traded_quantity': 2, 'average_traded_price': 1478.81, 'volume_traded': 5811351, 'total_buy_quantity': 433647, 'total_sell_quantity': 498933, 'ohlc': {'open': 1473.0, 'high': 1485.5, 'low': 1471.5, 'close': 1485.4}, 'change': -0.30968089403528587, 'last_trade_time': datetime.datetime(2025, 7, 16, 12, 58, 33), 'oi': 0, 'oi_day_high': 0, 'oi_day_low': 0, 'exchange_timestamp': datetime.datetime(2025, 7, 16, 12, 58, 34), 'depth': {'buy': [{'quantity': 1078, 'price': 1480.8, 'orders': 10}, {'quantity': 604, 'price': 1480.7, 'orders': 4}, {'quantity': 1687, 'price': 1480.6, 'orders': 9}, {'quantity': 1325, 'price': 1480.5, 'orders': 10}, {'quantity': 307, 'price': 1480.4, 'orders': 8}], 'sell': [{'quantity': 2440, 'price': 1481.0, 'orders': 9}, {'quantity': 550, 'price': 1481.1, 'orders': 3}, {'quantity': 559, 'price': 1481.2, 'orders': 7}, {'quantity': 1150, 'price': 1481.3, 'orders':

INFO:root:Ticks: [{'tradable': True, 'mode': 'full', 'instrument_token': 738561, 'last_price': 1480.8, 'last_traded_quantity': 36, 'average_traded_price': 1478.81, 'volume_traded': 5811546, 'total_buy_quantity': 434200, 'total_sell_quantity': 498873, 'ohlc': {'open': 1473.0, 'high': 1485.5, 'low': 1471.5, 'close': 1485.4}, 'change': -0.30968089403528587, 'last_trade_time': datetime.datetime(2025, 7, 16, 12, 58, 42), 'oi': 0, 'oi_day_high': 0, 'oi_day_low': 0, 'exchange_timestamp': datetime.datetime(2025, 7, 16, 12, 58, 42), 'depth': {'buy': [{'quantity': 1116, 'price': 1480.8, 'orders': 7}, {'quantity': 1319, 'price': 1480.7, 'orders': 9}, {'quantity': 409, 'price': 1480.6, 'orders': 6}, {'quantity': 1311, 'price': 1480.5, 'orders': 10}, {'quantity': 1042, 'price': 1480.4, 'orders': 12}], 'sell': [{'quantity': 2339, 'price': 1481.0, 'orders': 9}, {'quantity': 550, 'price': 1481.1, 'orders': 3}, {'quantity': 30, 'price': 1481.2, 'orders': 1}, {'quantity': 1394, 'price': 1481.3, 'orders'

INFO:root:Ticks: [{'tradable': True, 'mode': 'full', 'instrument_token': 738561, 'last_price': 1481.0, 'last_traded_quantity': 1, 'average_traded_price': 1478.81, 'volume_traded': 5811718, 'total_buy_quantity': 434117, 'total_sell_quantity': 498886, 'ohlc': {'open': 1473.0, 'high': 1485.5, 'low': 1471.5, 'close': 1485.4}, 'change': -0.29621650733809685, 'last_trade_time': datetime.datetime(2025, 7, 16, 12, 58, 48), 'oi': 0, 'oi_day_high': 0, 'oi_day_low': 0, 'exchange_timestamp': datetime.datetime(2025, 7, 16, 12, 58, 48), 'depth': {'buy': [{'quantity': 76, 'price': 1480.9, 'orders': 2}, {'quantity': 2240, 'price': 1480.8, 'orders': 7}, {'quantity': 41, 'price': 1480.7, 'orders': 6}, {'quantity': 535, 'price': 1480.6, 'orders': 7}, {'quantity': 1311, 'price': 1480.5, 'orders': 10}], 'sell': [{'quantity': 2192, 'price': 1481.0, 'orders': 10}, {'quantity': 325, 'price': 1481.1, 'orders': 2}, {'quantity': 259, 'price': 1481.2, 'orders': 3}, {'quantity': 997, 'price': 1481.3, 'orders': 8},

INFO:root:Ticks: [{'tradable': True, 'mode': 'full', 'instrument_token': 738561, 'last_price': 1480.9, 'last_traded_quantity': 1, 'average_traded_price': 1478.81, 'volume_traded': 5812030, 'total_buy_quantity': 434099, 'total_sell_quantity': 499498, 'ohlc': {'open': 1473.0, 'high': 1485.5, 'low': 1471.5, 'close': 1485.4}, 'change': -0.3029487006866837, 'last_trade_time': datetime.datetime(2025, 7, 16, 12, 58, 53), 'oi': 0, 'oi_day_high': 0, 'oi_day_low': 0, 'exchange_timestamp': datetime.datetime(2025, 7, 16, 12, 58, 54), 'depth': {'buy': [{'quantity': 507, 'price': 1480.9, 'orders': 3}, {'quantity': 962, 'price': 1480.8, 'orders': 4}, {'quantity': 530, 'price': 1480.7, 'orders': 7}, {'quantity': 1690, 'price': 1480.6, 'orders': 10}, {'quantity': 1284, 'price': 1480.5, 'orders': 8}], 'sell': [{'quantity': 2243, 'price': 1481.0, 'orders': 9}, {'quantity': 494, 'price': 1481.1, 'orders': 4}, {'quantity': 803, 'price': 1481.2, 'orders': 8}, {'quantity': 1193, 'price': 1481.3, 'orders': 10

INFO:root:Ticks: [{'tradable': True, 'mode': 'full', 'instrument_token': 738561, 'last_price': 1480.7, 'last_traded_quantity': 3, 'average_traded_price': 1478.81, 'volume_traded': 5813790, 'total_buy_quantity': 428696, 'total_sell_quantity': 501405, 'ohlc': {'open': 1473.0, 'high': 1485.5, 'low': 1471.5, 'close': 1485.4}, 'change': -0.3164130873838727, 'last_trade_time': datetime.datetime(2025, 7, 16, 12, 58, 58), 'oi': 0, 'oi_day_high': 0, 'oi_day_low': 0, 'exchange_timestamp': datetime.datetime(2025, 7, 16, 12, 58, 59), 'depth': {'buy': [{'quantity': 527, 'price': 1480.7, 'orders': 5}, {'quantity': 409, 'price': 1480.6, 'orders': 6}, {'quantity': 1200, 'price': 1480.5, 'orders': 7}, {'quantity': 7, 'price': 1480.4, 'orders': 1}, {'quantity': 1079, 'price': 1480.3, 'orders': 3}], 'sell': [{'quantity': 813, 'price': 1480.8, 'orders': 9}, {'quantity': 1413, 'price': 1480.9, 'orders': 12}, {'quantity': 3208, 'price': 1481.0, 'orders': 18}, {'quantity': 973, 'price': 1481.1, 'orders': 13}

### 6. Conclusion & Next Steps

Congratulations! You have successfully connected to the Kite WebSocket and streamed live market data. 

**Recap:**
- We initialized the `KiteTicker` with our credentials.
- We defined callback functions to handle connection events and incoming tick data.
- We subscribed to an instrument (Reliance) to start receiving its live price updates.

**Next Steps to Explore:**
- **Change the Mode**: Experiment with `ws.MODE_LTP` and `ws.MODE_QUOTE` to see how the data you receive changes. This is important for optimizing data usage.
- **Subscribe to More Instruments**: Find the instrument tokens for other stocks or indices and add them to the subscription list.
- **Add Reconnect Logic**: In the `on_close` function, you could add logic to automatically call `kws.connect()` again after a short delay to make your script more robust.